#### ============================================
###  1. Import Libraries
#### ============================================

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
import matplotlib.cm as cm


#### ============================================
###  2. Read a Data
#### ============================================

In [ ]:
Data = pd.read_csv("/Users/mac/Desktop/Zain Projects/Task 1 Salary Dataset/data/Cleaned_Salaries.csv")

#### ============================================
###  3. Check a Data
#### ============================================

In [ ]:
Data.head()

In [ ]:
Data.isnull().sum()

In [ ]:
Data.columns

#### ============================================
###  4. Exploratory Data Analysis
#### ============================================

In [ ]:
for col in ['BasePay','OvertimePay','OtherPay','Benefits','TotalPay','TotalPayBenefits']:
    Data[col+'_orig'] = Data[col].apply(lambda x: np.exp(x) if x > 0 else 0)

###  4.1 Avg Total Pay Trend

In [ ]:
yearly = Data.groupby('Year')['TotalPay_orig'].mean()

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#f0f2f5')
ax.set_facecolor('#f0f2f5')

ax.plot(yearly.index, yearly.values, 
        marker='o', color='#2E4057', linewidth=2.5, markersize=8)

for year, value in yearly.items():
    ax.annotate(f'${value/1000:.1f}K',
                xy=(year, value),
                xytext=(0, 10),
                textcoords='offset points',
                ha='center', fontsize=10, fontweight='bold', color='#2E4057')

ax.set_title('Avg Total Pay by Year', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Avg Total Pay', fontsize=11)
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))
ax.set_xticks([2011, 2012, 2013, 2014])
ax.grid(True, alpha=0.4)

for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

### Insight: Salaries were on an upward trend for 3 years, but 2014 saw a dip — this could indicate budget cuts or hiring of lower-paid employees that year. It tells us compensation isn't always increasing linearly in public sector organizations. ###

###  4.2 Distribution of Total Pay 

In [ ]:
TEAL      = "#2AB3A3"
DARK_BLUE = "#2C3E6B"
LIGHT_BLUE = "#87CEEB"
ORANGE    = "#F5A623"
BG        = "#EEF2F5"
 
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   BG,
    "axes.spines.top":  False,
    "axes.spines.right": False,
    "axes.grid":        True,
    "grid.color":       "white",
    "grid.linewidth":   0.9,
    "font.size":        12,
})
 
def k_fmt(x, _):
    return f"{int(x/1000)}K"
 
def k_fmt_dollar(x, _):
    return f"${int(x/1000)}K"

In [ ]:
pay_k = Data[Data['TotalPay_orig'] > 0]['TotalPay_orig'] / 1000
 
mean_k   = pay_k.mean()
median_k = pay_k.median()
 
fig, ax = plt.subplots(figsize=(14, 7))
ax.hist(pay_k, bins=60, color=TEAL, edgecolor="white", linewidth=0.4)
 
ax.axvline(mean_k,   color="#E05555", linestyle="--", linewidth=2,
           label=f"Mean: ${mean_k:.0f}K")
ax.axvline(median_k, color="#E8A030", linestyle="--", linewidth=2,
           label=f"Median: ${median_k:.0f}K")
 
ax.set_title("Distribution of Total Pay", fontsize=15, fontweight="bold", pad=14)
ax.set_xlabel("Total Pay ($K)", fontsize=12)
ax.set_ylabel("Number of Employees", fontsize=12)
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.legend(frameon=True, fontsize=11, loc="upper right")
 
plt.tight_layout()
plt.savefig("dist_total_pay.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ حُفظت: dist_total_pay.png")

### Insight: The salary distribution is not normal — it's right-skewed, meaning a small group of very high earners is pulling the average up. The median is a more honest representation of what a typical employee earns. This is classic in public sector data. ###

###  4.3 Overtime vs Base Pay

In [ ]:
sample = Data[
    (Data['OvertimePay_orig'] > 0) & (Data['BasePay_orig'] > 0)
].sample(3000, random_state=42)
 
years  = sorted(sample['Year'].unique())
norm   = plt.Normalize(vmin=min(years), vmax=max(years))
colors = cm.Blues(norm(sample['Year']))
 
fig, ax = plt.subplots(figsize=(11, 7))
sc = ax.scatter(
    sample['BasePay_orig'] / 1000,
    sample['OvertimePay_orig'] / 1000,
    c=sample['Year'], cmap='Blues',
    alpha=0.5, s=15, norm=norm
)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("Year", rotation=270, labelpad=15)
cbar.set_ticks(years)
cbar.set_ticklabels(years)
 
ax.set_title("Overtime vs Base Pay (OT earners only)", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Base Pay ($K)", fontsize=12)
ax.set_ylabel("Overtime Pay ($K)", fontsize=12)
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"${int(x)}K"))
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"${int(x)}K"))
 
plt.tight_layout()
plt.show()

### Insight: Overtime is not driven by salary level — it's driven by job type. Roles like police, firefighters, and nurses work 24/7 operations so they accumulate high overtime regardless of their base pay grade. This tells us that operational sector matters more than seniority when it comes to overtime. ###

###  4.4 Top 10 Paying Jobs

In [ ]:
top_jobs = Data.groupby('JobTitle').filter(lambda x: len(x) >= 50)
top10 = (
    top_jobs.groupby('JobTitle')['TotalPay_orig']
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .sort_values(ascending=True)
) / 1000
 
bar_colors = [DARK_BLUE if i >= 7 else TEAL for i in range(len(top10))]
 
fig, ax = plt.subplots(figsize=(14, 8))
bars = ax.barh(top10.index, top10.values, color=bar_colors, height=0.6)
 
for bar, val in zip(bars, top10.values):
    ax.text(bar.get_width() + 1.5,
            bar.get_y() + bar.get_height() / 2,
            f"${val:.0f}K",
            va="center", ha="left", fontsize=11, fontweight="bold")
 
ax.set_title("Top 10 Highest Paying Job Titles\n(min 50 employees)",
             fontsize=15, fontweight="bold", pad=14)
ax.set_xlabel("Avg Total Pay ($K)", fontsize=12)
ax.set_xlim(0, top10.max() * 1.18)
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{int(x)}K"))
ax.tick_params(axis='y', labelsize=11)
ax.spines['left'].set_visible(False)
ax.tick_params(axis='y', length=0)
 
plt.tight_layout()
plt.show()

### Insight: The highest-paid roles are in Fire & Legal departments — not in tech or management. Battalion Chief averages $252K total pay, which is 3x the city average. This shows that specialized emergency and legal expertise commands a significant premium in the public sector. ###

###  4.5 Pay Components Stacked Bar

In [ ]:
comp = Data.groupby('Year')[
    ['BasePay_orig', 'OvertimePay_orig', 'OtherPay_orig', 'Benefits_orig']
].mean()
 
comp.columns = ['Base Pay', 'Overtime', 'Other Pay', 'Benefits']
stack_colors = [DARK_BLUE, LIGHT_BLUE, ORANGE, TEAL]
 
fig, ax = plt.subplots(figsize=(10, 7))
bottom = np.zeros(len(comp))
for col, color in zip(comp.columns, stack_colors):
    ax.bar(comp.index.astype(str), comp[col], bottom=bottom,
           label=col, color=color, edgecolor="white", width=0.55)
    bottom += comp[col].values
 
ax.set_title("Avg Pay Components by Year", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Avg Pay ($K)", fontsize=12)
ax.yaxis.set_major_formatter(FuncFormatter(k_fmt_dollar))
ax.legend(loc="upper left", fontsize=10, frameon=True)
ax.tick_params(axis='x', rotation=0)
 
plt.tight_layout()
plt.show()

### Insight: The real cost of an employee to the city is significantly higher than just their salary. When you include benefits, the average total compensation reaches $93.7K versus a base of $66.3K. For workforce planning, you always need to account for this gap — it's nearly a 40% overhead on top of base pay.

###  4.6 Receiving OT

In [ ]:
ot_pct = Data.groupby('Year')['OvertimePay_orig'].apply(
    lambda x: (x > 0).mean() * 100
)
 
fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.bar(ot_pct.index.astype(str), ot_pct.values,
              color=LIGHT_BLUE, edgecolor="white", width=0.55)
 
# القيم فوق كل بار
for bar, val in zip(bars, ot_pct.values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.3,
            f"{val:.1f}%",
            ha="center", va="bottom", fontsize=11, fontweight="bold")
 
ax.set_title("% Employees Receiving Overtime Pay", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("% Employees", fontsize=12)
ax.set_ylim(0, 60)
ax.tick_params(axis='x', rotation=0)
 
plt.tight_layout()
plt.show() 

### Insight: Overtime is not spread evenly across the workforce — it's concentrated in specific departments. More than half of employees receive zero overtime. The gradual increase year-over-year suggests either growing operational demand or understaffing in key departments, forcing existing staff to work more hours.

###  4.7 Boxplot by Year

In [ ]:
years_sorted = sorted(Data['Year'].unique())
pay_by_year  = [Data[Data['Year'] == y]['TotalPay_orig'].dropna() / 1000
                for y in years_sorted]
 
box_colors = ["#607080", TEAL, LIGHT_BLUE, ORANGE]
 
fig, ax = plt.subplots(figsize=(10, 7))
bp = ax.boxplot(
    pay_by_year,
    labels=years_sorted,
    patch_artist=True,
    showfliers=True,
    flierprops=dict(marker='o', markersize=2, alpha=0.3, color='gray'),
    medianprops=dict(color='white', linewidth=2),
    whiskerprops=dict(color='#555555'),
    capprops=dict(color='#555555'),
)
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
 
ax.set_title("Total Pay Distribution by Year", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Total Pay ($K)", fontsize=12)
ax.yaxis.set_major_formatter(FuncFormatter(k_fmt_dollar))
 
plt.tight_layout()
plt.show()

### Insight: The salary structure is very stable year over year — the middle 50% of employees earn within a consistent range. However, extreme outliers exist every year, likely senior officials or employees with massive overtime accumulation. These outliers are what inflate the mean and make it misleading as a central tendency measure.

###  4.8 Most Common Job Titles

In [ ]:
top_titles = (Data['JobTitle'].value_counts()
              .head(10)
              .sort_values(ascending=True))
 
fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(top_titles.index, top_titles.values,
               color=ORANGE, height=0.65, edgecolor="white")
 
for bar, val in zip(bars, top_titles.values):
    ax.text(bar.get_width() + 30,
            bar.get_y() + bar.get_height() / 2,
            f"{val:,}",
            va="center", ha="left", fontsize=10, fontweight="bold")
 
ax.set_title("Top 10 Most Common Job Titles", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Number of Records", fontsize=12)
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.spines['left'].set_visible(False)
ax.tick_params(axis='y', length=0, labelsize=10)
 
plt.tight_layout()
plt.show()

### Insight: The workforce is dominated by Transit and Healthcare roles — together they represent a large portion of the headcount. This directly explains why overtime is so prevalent: both sectors run around the clock, 365 days a year. It also means any salary policy change would have a massive impact given the volume of employees in these roles.